<a href="https://colab.research.google.com/github/junseok-jay/guitar_CV/blob/main/test_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!pip install -q ultralytics
!pip install roboflow

Thu Mar 26 14:15:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [23]:
from google.colab import userdata
KEY = userdata.get('RF_KEY')

from roboflow import Roboflow
rf = Roboflow(api_key=KEY)
project = rf.workspace("s-workspace-y3mjn").project("guitar-neck-detection-suhgk-uemtb")

dataset_dir = '/content/datasets'
project.version(3).download("yolov11", location=dataset_dir)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/datasets in yolov11:: 100%|██████████| 2016/2016 [00:00<00:00, 5700.74it/s]


In [24]:
from pathlib import Path

yaml_path = Path(f"{dataset_dir}/data.yaml")
text = yaml_path.read_text(encoding="utf-8")
text = text.replace("train: ../train/images", "train: train/images")
text = text.replace("val: ../valid/images", "val: valid/images")
text = text.replace("test: ../test/images", "test: test/images")
yaml_path.write_text(text, encoding="utf-8")
print(yaml_path.read_text(encoding="utf-8"))

train: train/images
val: valid/images
test: test/images

nc: 1
names: ['neck']

roboflow:
  workspace: s-workspace-y3mjn
  project: guitar-neck-detection-suhgk-uemtb
  version: 3
  license: CC BY 4.0
  url: https://universe.roboflow.com/s-workspace-y3mjn/guitar-neck-detection-suhgk-uemtb/dataset/3


In [25]:
from ultralytics import YOLO

model = YOLO("yolo11n-seg.pt")
model.train(
    data=yaml_path,
    epochs=80,
    imgsz=640,
    batch=8,
    project="runs/guitar_neck_seg",
    name="train"
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.28 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d097c0a5e50>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [26]:
from ultralytics import YOLO

best_model = YOLO("runs/segment/runs/guitar_neck_seg/train/weights/best.pt")
metrics = best_model.val(data="datasets/data.yaml")
print(metrics)

Ultralytics 8.4.28 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-seg summary (fused): 114 layers, 2,834,763 parameters, 0 gradients, 9.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1778.4±527.7 MB/s, size: 55.2 KB)
val: Scanning /content/datasets/valid/labels.cache... 84 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 84/84 39.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.1it/s 2.9s
                   all         84         84      0.999          1      0.995      0.973      0.999          1      0.995      0.951
Speed: 6.9ms preprocess, 8.1ms inference, 0.0ms loss, 4.2ms postprocess per image
Results saved to /content/runs/segment/val
ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.Confus

In [27]:
!pwd
!find runs/segment/runs/guitar_neck_seg/train/weights -maxdepth 2 -type f | sort

/content
runs/segment/runs/guitar_neck_seg/train/weights/best.pt
runs/segment/runs/guitar_neck_seg/train/weights/last.pt


In [28]:
from google.colab import files
files.download("runs/segment/runs/guitar_neck_seg/train/weights/best.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>